# The benchmark catalogue — ours and imported

The catalogue no longer holds only ScreamingFace-authored boards. Benchmarks imported from
[inspect_evals](https://ukgovernmentbeis.github.io/inspect_evals/) — GSM8K, MMLU, ARC, BoolQ and
friends — sit beside them as first-class boards: same listing, same `sf.evaluate(...)` call, same
report.

Every benchmark carries an **origin**, and the listing renders one group per origin, so you can
always tell what we built from what we brought in. An imported board keeps its source eval's own
scorer — the upstream grading logic is called, never reimplemented — and its dataset is snapshotted
and pinned at import time, so a published board never drifts under you.

This notebook walks the whole path: list the catalogue → read an imported board's card → run a
fusion against it.

## 0. Before running

Working from a checkout? `just local-stack-notebooks` in `packages/screamingface/` does every step
below — assets, stack, and Jupyter — in one command. Otherwise, from a terminal:

```bash
screamingface up      # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when finished.
Stack management stays outside the notebook so **Run All** never starts or stops local services.

**Where the imported boards live.** An Engine serves imported boards only when it runs with its
`inspect` extra (the upstream scorers come from `inspect-ai`, which cannot co-install with the
local runtime's dependencies — a declared conflict, not an accident). The local
`screamingface up` stack therefore lists the ScreamingFace group only. **An inspect-capable
Engine is a prerequisite for everything past section 1**: the listing works against any Engine,
and the first cell of section 2 checks for the imported shelf and stops with guidance rather
than failing partway through. To browse and run the imported catalogue, point the SDK at an
inspect-capable Engine before starting the kernel:

```bash
export SCREAMINGFACE_ENGINE_URL="https://<an-engine-with-the-inspect-extra>"
```

Leaving it unset falls back to a running local stack, then to the hosted default.

**Running one yourself, from a checkout.** The `just local-stack-notebooks` recipe above does
exactly this: it bakes the imported snapshots, serves an inspect-capable Engine beside the
`screamingface up` stack on a free port — reaching the same Gateway on `:9105`, so only the
Engine URL changes — and exports `SCREAMINGFACE_ENGINE_URL` for the kernel it opens.

In [ ]:
import screamingface as sf

sf.connect()

## 1. List the catalogue — one group per origin

The listing groups by each benchmark's `origin`: a **ScreamingFace** tab for our boards, first,
and an **inspect_evals** tab linking to the source collection. Without `ipywidgets` the same
grouping renders as titled sections. The search box filters rows inside every group.

In [ ]:
benchmarks = sf.benchmarks.list()
benchmarks

## 2. Read an imported board's card

Imported boards are named `inspect-<key>` after their upstream eval. We'll use **GSM8K** —
grade-school math word problems, graded by the eval's own numeric match against the published
answer. That grading is free: no judge, no grading tokens, so cost is answer generation only.

The card carries the provenance: `origin` names the source collection, and `revision` pins the
imported dataset snapshot — two catalogues showing the same revision asked the exact same
questions.

The first lines below are the gate from section 0: if this Engine serves no imported boards,
the notebook stops here with directions instead of failing on the lookup.

In [ ]:
if not any(benchmark.origin == "inspect_evals" for benchmark in benchmarks):
    raise RuntimeError(
        "this Engine serves no imported boards — the rest of this notebook needs an "
        "Engine running with its inspect extra; see section 0 for how to point "
        "SCREAMINGFACE_ENGINE_URL at one"
    )

gsm8k = sf.benchmarks.get("inspect-gsm8k")
{
    "id": gsm8k.id,
    "title": gsm8k.title,
    "origin": gsm8k.origin,
    "revision": gsm8k.revision,
    "case_count": gsm8k.case_count,
}

## 3. Run a fusion against it

An imported board takes a fusion exactly like a home-grown one — the Engine invokes the candidate
as an opaque recipe, so nothing about the import changes how ensembles run. `limit` keeps the
rehearsal cheap; the calls below are paid model calls, so run this cell deliberately and rehearse
small before any full sweep.

In [ ]:
PANEL_PARAMS = {"max_tokens": 8192, "temperature": 0.0}
SYNTHESIS_PROMPT = (
    "You are given several models' step-by-step solutions to a grade-school math word "
    "problem. Check each chain of arithmetic, resolve any disagreement by re-deriving the "
    "disputed step, and answer with the single final number."
)

member1 = sf.Model(model="openrouter/qwen/qwen3.7-flash", params=PANEL_PARAMS)
member2 = sf.Model(model="openrouter/google/gemini-3.8-flash", params=PANEL_PARAMS)
synth = sf.Model(
    model="openrouter/anthropic/claude-haiku-4.5", params=PANEL_PARAMS, prompt=SYNTHESIS_PROMPT
)
math_panel = sf.Fusion(name="math_panel", members=[member1, member2], synthesizer=synth)

math_panel

In [ ]:
report = sf.evaluate(math_panel, benchmark="inspect-gsm8k", limit=2)
report

## 4. Read the per-case outcomes

Each case is one bit — the committed number matched the key or it did not — and the check row
carries what the candidate answered against what was expected, so a miss can be inspected rather
than just counted.

In [ ]:
for case in report.candidates.only.cases:
    grade = case.grade
    print(case.case_id, case.status, grade.score if grade else None)

## 5. A multiple-choice board — same calls, one thing to change

`inspect-mmlu` is 57 subjects of four-option questions, graded by the eval's `choice`
scorer against the published letter. The SDK calls are identical to section 3; what has to
change is the **synthesiser's prompt**, because the answer format did. Asking for "the
single final number" on a board that wants `A`/`B`/`C`/`D` is how a panel scores zero
while answering correctly.

Worth knowing about MCQ boards: a fusion has less to do here than on free text. A choice is
one discrete token, so the synthesiser cannot *merge* partial credit the way it can on a
rubric board — it can only weigh votes and pick. The two families are not comparable in
what they ask of an ensemble.

In [ ]:
MCQ_SYNTHESIS_PROMPT = (
    "You are given several experts' analyses of one multiple-choice question. Weigh their "
    "reasoning and the evidence they cite — not merely how many chose each option — and "
    "answer with the single best choice."
)

mcq_synth = sf.Model(
    model="openrouter/anthropic/claude-haiku-4.5",
    params=PANEL_PARAMS,
    prompt=MCQ_SYNTHESIS_PROMPT,
)
mcq_panel = sf.Fusion(name="mcq_panel", members=[member1, member2], synthesizer=mcq_synth)

mmlu_report = sf.evaluate(mcq_panel, benchmark="inspect-mmlu", limit=2)
mmlu_report

## 6. A yes/no board

`inspect-boolq` asks a reading-comprehension question whose answer is `Yes` or `No`, graded
by the eval's `pattern` scorer against a regex anchored at the end of the reply. That anchor
is the whole trick: a model that reasons for a paragraph and finishes with "Yes" scores,
while one that opens with "Yes, because…" does not. Say so in the prompt.

This board is free text rather than a fixed set of options, so unlike the MCQ boards it
carries a **check surface** — the mid-run pass/fail signal a `corrective_loop` reads (see
`09_corrective_loops.ipynb`). MCQ boards are refused one deliberately: pass/fail feedback
over four options is an elimination attack, not a hint.

In [ ]:
BOOLQ_SYNTHESIS_PROMPT = (
    "You are given several experts' readings of one passage and a yes/no question about it. "
    "Weigh their reasoning, then end your reply with exactly one word — Yes or No — as the "
    "final word, with nothing after it."
)

boolq_synth = sf.Model(
    model="openrouter/anthropic/claude-haiku-4.5",
    params=PANEL_PARAMS,
    prompt=BOOLQ_SYNTHESIS_PROMPT,
)
boolq_panel = sf.Fusion(name="boolq_panel", members=[member1, member2], synthesizer=boolq_synth)

boolq_report = sf.evaluate(boolq_panel, benchmark="inspect-boolq", limit=2)
boolq_report

In [ ]:
for case in boolq_report.candidates.only.cases:
    grade = case.grade
    print(case.case_id, case.status, grade.score if grade else None)

## 7. Export a run in inspect's own log format

An imported board's exam came from inspect_evals, so it is reasonable to want the *result*
in inspect's shape too. `report.export(format="inspect")` writes one Candidate's run as a
`.eval` log — the same format `inspect view` reads — carrying our per-case records and the
run's real metered cost.

It is a **one-way copy**, deliberately. The report you see above stays the record: the
leaderboard is never published from an inspect log, and nothing is ever read back in.

Writing that file needs `inspect-ai`, which `just local-stack-notebooks` installs into this
kernel for you. It is not present in *every* environment, though: `inspect-ai` and the SDK's
`runtime` extra — the local stack's own servers — have genuinely conflicting dependency pins,
so one environment cannot boot the stack **and** hold the exporter. The recipe sidesteps that
by keeping them apart rather than choosing: the Gateway and Scoreboard start from their own
environment, and this kernel gets the exporter.

If you are in a kernel without it, install `screamingface[inspect]` there and run the
evaluation again in that session. Note there is no way to export from a saved file —
`report.to_json()` is a one-way copy and nothing reads it back — so the export has to happen
in the session that produced the run.

In [ ]:
from importlib.util import find_spec

if find_spec("inspect_ai") is None:
    print('no inspect-ai in this kernel — install "screamingface[inspect]" here and')
    print("re-run the evaluation: a Report cannot be rebuilt from a saved report.json")
else:
    eval_path = report.export("gsm8k-fusion.eval", format="inspect")
    print("wrote", eval_path)

## 8. What you can do with the `.eval` file

**There is nowhere to submit it.** Neither inspect_ai nor inspect_evals runs a service that
accepts uploaded logs, and inspect_evals publishes no results leaderboard — its site is a
catalogue of eval *implementations*, not of scores. Anyone telling you to upload a log
somewhere is describing a mechanism that does not exist.

What the file is genuinely for is **viewing and sharing**, all self-hosted:

```bash
inspect view --log-dir .                      # open the local viewer on this file
inspect view bundle --log-dir . --output-dir logs-www   # a self-contained static site
inspect log convert --to json --output-dir . run.eval   # the same log as JSON
```

`inspect view bundle` bakes the logs and a copy of the viewer into a directory you can put
on any static host, which is the closest thing to sharing a run with someone. Inspect can
also read and write logs straight from object storage — point `INSPECT_LOG_DIR` at an
`s3://`, `gcs://`, `az://` or `hf://` URL — so a team can keep a shared log directory
without any service in between.

Numbers reach an inspect_evals README the slow way: a GitHub pull request updating that
eval's own README table. Their contributing guide allows self-reported results provided
they are public, specific, and labelled as self-reported rather than presented as
apples-to-apples official scores — which matters here, because a fusion of several models
is not the single-model setup those tables usually quote.

To publish a result *in our product*, use the leaderboard instead — see
`sf.leaderboards.submit(...)` in `07_ifeval.ipynb`. That path is ours end to end, and it is
never fed from an inspect log.

## 9. The rest of the shelf

Three boards, three scorer families, one set of calls — that is the whole point of the
import. The remaining seven work the same way; pick an id from the inspect_evals group in
section 1 and match the synthesiser's prompt to how that board is graded:

- **A final number**, graded by numeric match — `inspect-gsm8k`.
- **A letter**, graded by the `choice` scorer — `inspect-mmlu`, `inspect-mmlu_pro`,
  `inspect-arc_easy`, `inspect-arc_challenge`, `inspect-commonsense_qa`,
  `inspect-winogrande`, `inspect-race_h`.
- **Yes / No as the last word**, graded by an anchored pattern — `inspect-boolq`.
- **yes / no anywhere in the reply**, graded by `includes` — `inspect-paws`.

A `limit=N` run is a smoke test, not a ranking: on small subsamples a point or two between
two systems is noise. Run the whole set before quoting a comparison, and read `coverage`
beside the score — these boards score the gradeable subset and publish how much of the run
that was, so a good score over thin coverage is a formatting failure wearing a knowledge
result's clothes.